In [14]:
from pathlib import Path
import shutil
import subprocess
from collections import Counter

### Step 1. Input and class setup (World Reference Base 2006 Soil Groups)

Input raster:
- `release/2006/data/world_reference_base_2006_soil_groups.tif`

Class IDs follow the SoilGrids WRB 2006 coding.
Use visual tiles for map rendering and value tiles for class decoding (`class_code = R`).

In [15]:
base = Path("data")
in_tif = base / "world_reference_base_2006_soil_groups.tif"
out_dir = Path("out/world_reference_base_2006_soil_groups_2006")
cog_tif = out_dir / "world_reference_base_2006_soil_groups_2006_cog.tif"
visual_tiles_dir = out_dir / "tiles_visual"
value_tiles_dir = out_dir / "tiles_values"

visual_colors_txt = base / "world_reference_base_2006_soil_groups_colors.txt"
value_colors_txt = base / "world_reference_base_2006_soil_groups_value_encoding_colors.txt"

# Official SoilGrids WRB MostProbable mapping (MostProbable.rat.xml)
class_meta = {
    0: ("AC", "Acrisols"),
    1: ("AB", "Albeluvisols"),
    2: ("AL", "Alisols"),
    3: ("AN", "Andosols"),
    4: ("AR", "Arenosols"),
    5: ("CL", "Calcisols"),
    6: ("CM", "Cambisols"),
    7: ("CH", "Chernozems"),
    8: ("CR", "Cryosols"),
    9: ("DU", "Durisols"),
    10: ("FR", "Ferralsols"),
    11: ("FL", "Fluvisols"),
    12: ("GL", "Gleysols"),
    13: ("GY", "Gypsisols"),
    14: ("HS", "Histosols"),
    15: ("KS", "Kastanozems"),
    16: ("LP", "Leptosols"),
    17: ("LX", "Lixisols"),
    18: ("LV", "Luvisols"),
    19: ("NT", "Nitisols"),
    20: ("PH", "Phaeozems"),
    21: ("PL", "Planosols"),
    22: ("PT", "Plinthosols"),
    23: ("PZ", "Podzols"),
    24: ("RG", "Regosols"),
    25: ("SC", "Solonchaks"),
    26: ("SN", "Solonetz"),
    27: ("ST", "Stagnosols"),
    28: ("UM", "Umbrisols"),
    29: ("VR", "Vertisols"),
}

out_dir.mkdir(parents=True, exist_ok=True)

In [12]:
# Quick data check: list class IDs present in this AOI (no Python GDAL bindings required)
xyz = subprocess.run(
    ["gdal_translate", "-of", "XYZ", str(in_tif), "/vsistdout/"],
    check=True,
    capture_output=True,
    text=True,
)

counts = Counter()
for line in xyz.stdout.splitlines():
    parts = line.split()
    if len(parts) < 3:
        continue
    # XYZ rows are: x y value
    value = int(float(parts[2]))
    counts[value] += 1

values = sorted(counts)
print("Class IDs present in raster:", values)
for v in values:
    code, label = class_meta.get(v, ("??", "Unknown"))
    print(f"{v:>2}  {code:<2}  {label:<15} pixels={counts[v]}")

Class IDs present in raster: [0, 4, 6, 12, 18, 19, 20, 21]
 0  AC  Acrisols        pixels=18165
 4  AR  Arenosols       pixels=55
 6  CM  Cambisols       pixels=54
12  GL  Gleysols        pixels=172
18  LV  Luvisols        pixels=57
19  NT  Nitisols        pixels=46
20  PH  Phaeozems       pixels=95
21  PL  Planosols       pixels=3550


### Step 2. Convert to COG and generate tiles

In [16]:
# Convert categorical raster to COG
subprocess.run([
    "gdal_translate", str(in_tif), str(cog_tif),
    "-of", "COG",
    "-ot", "Int16",
    "-co", "COMPRESS=DEFLATE",
    "-co", "RESAMPLING=NEAREST",
    "-co", "OVERVIEWS=AUTO"
], check=True)
print("Created COG:", cog_tif)

# Visual tiles (legend colors)
colorized_tif = out_dir / "world_reference_base_2006_soil_groups_colorized.tif"
subprocess.run([
    "gdaldem", "color-relief", "-nearest_color_entry",
    str(cog_tif), str(visual_colors_txt), str(colorized_tif)
], check=True)

# Preflight gdal2tiles runtime
gdal2tiles = shutil.which("gdal2tiles.py")
if not gdal2tiles:
    raise RuntimeError("gdal2tiles.py not found in PATH. Install GDAL first (e.g., `brew install gdal`).")

gdal2tiles_python = None
with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()
if first_line.startswith("#!"):
    parts = first_line[2:].split()
    if parts:
        if parts[0].endswith("env") and len(parts) > 1:
            gdal2tiles_python = shutil.which(parts[1])
        else:
            gdal2tiles_python = parts[0]
if not gdal2tiles_python:
    gdal2tiles_python = shutil.which("python3") or shutil.which("python")
if not gdal2tiles_python:
    raise RuntimeError("Could not determine Python interpreter for gdal2tiles.py")
subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)

visual_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(colorized_tif), str(visual_tiles_dir)
], check=True)
print("Visual tiles written to:", visual_tiles_dir)

# Value tiles: encode class id in RGB (class_code = R)
value_encoded_tif = out_dir / "world_reference_base_2006_soil_groups_value_encoded.tif"
subprocess.run([
    "gdaldem", "color-relief", "-nearest_color_entry",
    str(cog_tif), str(value_colors_txt), str(value_encoded_tif)
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
    str(value_encoded_tif), str(value_tiles_dir)
], check=True)
print("Value tiles written to:", value_tiles_dir)
print("Decode in app with: class_code = R")

Input file size is 137, 162
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG: out/world_reference_base_2006_soil_groups_2006/world_reference_base_2006_soil_groups_2006_cog.tif
0...10...20...30...40...50...60...70...80...90...100 - done.


Warning 1: Input dataset has no nodata value. Ignoring 'nv' entry in color palette
Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0

Generating Overview Tiles:


...10...20...30...40...50...60...70...80...90...100 - done.
Visual tiles written to: out/world_reference_base_2006_soil_groups_2006/tiles_visual


...

0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:05.
0..

Generating Overview Tiles:


.10...20...30...40...50...60...70...80...90...100 - done.
Value tiles written to: out/world_reference_base_2006_soil_groups_2006/tiles_values
Decode in app with: class_code = R
